# agent_rnaseq — End-to-End Demo

This notebook runs a complete bulk RNA-seq pipeline on the synthetic demo data in `data/`.
It uses **dry-run mode** (`dry_run=True`), which routes through all pipeline stages via
the LangGraph orchestrator **without** invoking the actual bioinformatics tools (STAR,
DESeq2, etc.). Realistic mock results are then generated and loaded into the Streamlit
dashboard for interactive exploration.

## What you'll see

| Step | What happens |
|---|---|
| 1 | Verify synthetic FASTQ files in `data/` |
| 2 | Initialise SQLite database |
| 3 | Register reference genome, project, samples |
| 4 | Dispatch pipeline via `OrchestratorAgent` (LangGraph routing) |
| 5 | Generate realistic mock DE, GSEA, and QC results |
| 6 | Interactive volcano plot and GSEA bubble chart inline |
| 7 | Write Streamlit data files and launch dashboard |

## Prerequisites

```bash
# From the project root:
pip install -e ".[dev]"
# Then launch:
jupyter lab  # or: jupyter notebook
```

> **No Docker required** for this notebook. The pipeline runs in-process using SQLite.
> To launch the Streamlit dashboard after running all cells, follow the instructions in the last cell.

## 0 — Environment setup

Must be the **first** Python cell. Sets environment variables before any `src` import
so `get_settings()` (which uses `@lru_cache`) picks them up correctly.

In [1]:
import os
import sys
from pathlib import Path

# Resolve project root (works whether CWD is repo root or notebooks/)
HERE = Path.cwd()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEMO_DIR    = PROJECT_ROOT / "demo_output"
DATA_DIR    = PROJECT_ROOT / "data"
DB_PATH     = DEMO_DIR / "agent_rnaseq_demo.db"
STREAM_DIR  = DEMO_DIR / "streamlit_data"
DEMO_DIR.mkdir(exist_ok=True)
STREAM_DIR.mkdir(parents=True, exist_ok=True)

# Set required env vars BEFORE importing any src module
os.environ["DATABASE_URL"]       = f"sqlite:///{DB_PATH}"
os.environ["OPENAI_API_KEY"]     = "sk-demo-placeholder-not-called-in-dry-run-mode"
os.environ["API_KEY_BOOTSTRAP"]  = "demo-bootstrap-key-agent-rnaseq-notebook"
os.environ["OUTPUT_ROOT"]        = str(DEMO_DIR)
os.environ["REDIS_URL"]          = "redis://localhost:6379/0"

print(f"Project root : {PROJECT_ROOT}")
print(f"Database     : {DB_PATH}")
print(f"Streamlit dir: {STREAM_DIR}")

Project root : /Users/junesong/Developer/agent_rnaseq
Database     : /Users/junesong/Developer/agent_rnaseq/demo_output/agent_rnaseq_demo.db
Streamlit dir: /Users/junesong/Developer/agent_rnaseq/demo_output/streamlit_data


## 1 — Verify FASTQ data

In [2]:
import gzip

# If files are missing, generate them now
fastq_files = sorted(DATA_DIR.glob("*.fastq.gz"))
if len(fastq_files) < 8:
    print("Generating synthetic FASTQ files...")
    import importlib.util, types
    spec = importlib.util.spec_from_file_location("generate_data", DATA_DIR / "generate_data.py")
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.generate(DATA_DIR)
    fastq_files = sorted(DATA_DIR.glob("*.fastq.gz"))

print(f"{'File':<32} {'Size':>8}  {'Reads':>8}  {'Read length':>11}")
print("-" * 65)
for fp in fastq_files:
    kb = fp.stat().st_size / 1024
    with gzip.open(fp, "rt") as fh:
        lines = [next(fh) for _ in range(4)]
    rlen = len(lines[1].strip())
    # count reads (each read = 4 lines)
    with gzip.open(fp, "rt") as fh:
        n_reads = sum(1 for _ in fh) // 4
    print(f"{fp.name:<32} {kb:>7.1f}K  {n_reads:>8,d}  {rlen:>11} bp")

File                                 Size     Reads  Read length
-----------------------------------------------------------------
ctrl_1_R1.fastq.gz                 207.0K     2,000          100 bp
ctrl_1_R2.fastq.gz                 206.8K     2,000          100 bp
ctrl_2_R1.fastq.gz                 206.8K     2,000          100 bp
ctrl_2_R2.fastq.gz                 207.0K     2,000          100 bp
treat_1_R1.fastq.gz                206.8K     2,000          100 bp
treat_1_R2.fastq.gz                206.8K     2,000          100 bp
treat_2_R1.fastq.gz                206.8K     2,000          100 bp
treat_2_R2.fastq.gz                206.8K     2,000          100 bp


## 2 — Initialise database

In [3]:
from sqlalchemy import create_engine, inspect
from sqlalchemy.orm import Session
from src.db.base import Base

# Import all model modules so SQLAlchemy registers their tables
import src.db.models.auth      # noqa: F401
import src.db.models.genome    # noqa: F401
import src.db.models.project   # noqa: F401
import src.db.models.run       # noqa: F401
import src.db.models.results   # noqa: F401

# Remove stale demo DB if it exists
DB_PATH.unlink(missing_ok=True)

engine = create_engine(
    f"sqlite:///{DB_PATH}",
    connect_args={"check_same_thread": False},
)
Base.metadata.create_all(engine)

tables = inspect(engine).get_table_names()
print(f"Database: {DB_PATH}")
print(f"Tables created ({len(tables)}): {', '.join(tables)}")

Database: /Users/junesong/Developer/agent_rnaseq/demo_output/agent_rnaseq_demo.db
Tables created (14): analysis_runs, api_keys, artifacts, deg_results, gsea_results, pipeline_stages, projects, qc_metrics, reference_genomes, run_samples, samples, scrna_cluster_results, splicing_results, variant_calls


## 3 — Register genome, project, and samples

In [4]:
import hashlib
import uuid

from src.db.enums import SampleType
from src.db.models.auth import APIKey
from src.db.models.genome import ReferenceGenome
from src.db.models.project import Project, Sample

SAMPLES_META = [
    ("ctrl_1",  "control",   1),
    ("ctrl_2",  "control",   2),
    ("treat_1", "treatment", 1),
    ("treat_2", "treatment", 2),
]

with Session(engine) as db:
    # Reference genome (mock paths — dry-run mode does not open these files)
    genome = ReferenceGenome(
        name="GRCh38_GENCODE_v43",
        species="homo_sapiens",
        build="GRCh38",
        annotation_version="GENCODE_v43",
        fasta_path="/data/genomes/GRCh38.fa",
        gtf_path="/data/genomes/GRCh38.gtf",
        star_index_path="/data/indexes/star_GRCh38",
        salmon_index_path="/data/indexes/salmon_GRCh38",
    )
    db.add(genome)
    db.flush()
    GENOME_ID   = str(genome.id)
    GENOME_NAME = genome.name  # capture before session closes

    # Project
    project = Project(
        name="synthetic_demo",
        description="Demo bulk RNA-seq: treatment vs. control (4 samples)",
        owner="demo_user",
    )
    db.add(project)
    db.flush()
    PROJECT_ID   = str(project.id)
    PROJECT_NAME = project.name  # capture before session closes

    # Samples
    SAMPLE_IDS: list[str] = []
    for name, condition, rep in SAMPLES_META:
        sample = Sample(
            project_id=project.id,
            name=name,
            sample_type=SampleType.bulk_rnaseq,
            condition=condition,
            replicate=rep,
            fastq_r1_path=str(DATA_DIR / f"{name}_R1.fastq.gz"),
            fastq_r2_path=str(DATA_DIR / f"{name}_R2.fastq.gz"),
            is_paired_end=True,
        )
        db.add(sample)
        db.flush()
        SAMPLE_IDS.append(str(sample.id))

    # API key (for future HTTP API use)
    raw_key = "demo-api-key-notebook-agent-rnaseq"
    api_key = APIKey(
        name="notebook",
        key_hash=hashlib.sha256(raw_key.encode()).hexdigest(),
        created_by="demo_user",
    )
    db.add(api_key)
    db.flush()
    API_KEY_ID = str(api_key.id)

    db.commit()

print(f"Genome  : {GENOME_ID}  ({GENOME_NAME})")
print(f"Project : {PROJECT_ID}  ({PROJECT_NAME})")
print(f"Samples : {len(SAMPLE_IDS)} registered")
for i, (name, cond, rep) in enumerate(SAMPLES_META):
    print(f"  [{i}] {name:10s}  condition={cond:10s}  replicate={rep}  id={SAMPLE_IDS[i][:8]}…")

Genome  : 571d43d3-a059-4e98-a2e9-5444a6a5613c  (GRCh38_GENCODE_v43)
Project : f0b3885f-0fa5-43c5-93b9-1c592b43f098  (synthetic_demo)
Samples : 4 registered
  [0] ctrl_1      condition=control     replicate=1  id=e9a02e50…
  [1] ctrl_2      condition=control     replicate=2  id=b537c517…
  [2] treat_1     condition=treatment   replicate=1  id=2d728ae7…
  [3] treat_2     condition=treatment   replicate=2  id=5b741498…


## 4 — Run the agent pipeline

`OrchestratorAgent.dispatch()` runs the **LangGraph** state machine:
- A **router node** reads `RunState.stages` and picks the next unfinished stage
- Each **stage node** is invoked in order: qc → alignment → quantification → differential_expression → gsea
- With `dry_run=True`, nodes update state without calling bioinformatics binaries

The LLM client is mocked so no OpenAI API calls are made.

In [5]:
import time
from datetime import datetime, timezone
from unittest.mock import MagicMock

from src.agents.orchestrator import OrchestratorAgent, RunConfig
from src.db.enums import (
    Aligner, AlignmentMode, Executor, PipelineType, RunStatus, StageName,
)
from src.db.models.project import RunSample
from src.db.models.run import AnalysisRun

STAGES = [
    StageName.qc,
    StageName.alignment,
    StageName.quantification,
    StageName.differential_expression,
    StageName.gsea,
]
CONTRAST = "treatment_vs_control"

# Create AnalysisRun DB record
with Session(engine) as db:
    run = AnalysisRun(
        project_id=uuid.UUID(PROJECT_ID),
        genome_id=uuid.UUID(GENOME_ID),
        created_by=uuid.UUID(API_KEY_ID),
        name="treatment_vs_control_demo",
        status=RunStatus.running,
        pipeline_type=PipelineType.bulk_rnaseq,
        alignment_mode=AlignmentMode.genome,
        aligner=Aligner.star,
        run_config={
            "stages":       [s.value for s in STAGES],
            "dry_run":      True,
            "de_contrasts": [{"name": CONTRAST,
                               "numerator": "treatment",
                               "denominator": "control"}],
        },
    )
    db.add(run)
    db.flush()
    for sid in SAMPLE_IDS:
        db.add(RunSample(run_id=run.id, sample_id=uuid.UUID(sid)))
    db.commit()
    db.refresh(run)
    RUN_ID = str(run.id)

print(f"Run created: {RUN_ID}")
print()

# Build RunConfig and dispatch through LangGraph
run_config = RunConfig(
    run_name="treatment_vs_control_demo",
    genome_id=GENOME_ID,
    pipeline_type=PipelineType.bulk_rnaseq,
    stages=STAGES,
    executor=Executor.local,
    dry_run=True,
)

orchestrator = OrchestratorAgent(llm_client=MagicMock(), dry_run=True)

t0 = time.perf_counter()
print("Dispatching pipeline via OrchestratorAgent (dry_run=True)...")
print()
final_state = orchestrator.dispatch(run_config, RUN_ID)
elapsed = time.perf_counter() - t0

print(f"{'─'*60}")
print(f"Pipeline complete in {elapsed:.3f}s")
print()
print(f"Stages planned  : {final_state.get('stages', [])}")
print(f"Stages completed: {final_state.get('completed_stages', [])}")
print(f"Failed stage    : {final_state.get('failed_stage') or 'none'}")

# Mark run as completed
with Session(engine) as db:
    r = db.get(AnalysisRun, uuid.UUID(RUN_ID))
    r.status = RunStatus.completed
    r.completed_at = datetime.now(timezone.utc)
    db.commit()
print()
print("Run status → completed")

Run created: 5a666a5b-8f4b-4878-990a-623eb85fd5bb

Dispatching pipeline via OrchestratorAgent (dry_run=True)...

────────────────────────────────────────────────────────────
Pipeline complete in 0.003s

Stages planned  : ['qc', 'alignment', 'quantification', 'differential_expression', 'gsea']
Stages completed: ['qc', 'alignment', 'quantification', 'differential_expression', 'gsea']
Failed stage    : none

Run status → completed


## 5 — Generate realistic mock results

In dry-run mode the orchestrator routes through stages but does not invoke bioinformatics
tools. Here we generate statistically realistic mock data that mirrors what DESeq2,
fgsea/Reactome, and FastQC would actually produce. This data drives the Streamlit
dashboard visualisations.

In [6]:
import json
import random

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
random.seed(42)

# ── DE Results: 300 genes (treatment vs. control) ─────────────────────────────
N_GENES = 300
GENE_NAMES = [
    "TP53","BRCA1","BRCA2","MYC","EGFR","KRAS","CDK4","RB1","PTEN","AKT1",
    "VEGFA","CCND1","CDH1","STAT3","HIF1A","BCL2","ERBB2","PIK3CA","MTOR",
    "AR","ESR1","MDM2","CTNNB1","NOTCH1","WNT5A","FOS","JUN","NF1","RAF1",
    "MAP2K1","CDKN2A","CDKN1A","PCNA","MCM2","Ki67","CCNE1","E2F1","AURKA",
    "PLK1","TOP2A","TYMS","DHFR","TS","FOXM1","CDC20","BUB1","MAD2L1",
    "BIRC5","SKP2","CKS1B","MYBL2","MKI67","UBE2C","CENPF","NUSAP1",
]

gene_ids   = [f"ENSG{i:011d}" for i in range(1, N_GENES + 1)]
gene_names = [GENE_NAMES[i - 1] if i <= len(GENE_NAMES) else random.choice(GENE_NAMES) + f"_{i}"
              for i in range(1, N_GENES + 1)]

baseMean = rng.exponential(400, N_GENES) + 20
log2fc   = rng.normal(0, 0.8, N_GENES)

# Spike in ~40 genuinely DE genes
de_idx_up   = rng.choice(N_GENES, 22, replace=False)
de_idx_down = rng.choice([i for i in range(N_GENES) if i not in de_idx_up], 18, replace=False)
log2fc[de_idx_up]   = rng.uniform(1.5, 4.0, len(de_idx_up))
log2fc[de_idx_down] = rng.uniform(-4.0, -1.5, len(de_idx_down))

lfcse = np.abs(log2fc) * rng.uniform(0.15, 0.35, N_GENES) + 0.08
stat  = log2fc / lfcse

pvalue = np.clip(np.abs(rng.normal(0.4, 0.3, N_GENES)), 0.001, 1.0)
pvalue[de_idx_up]   = rng.uniform(1e-15, 0.001, len(de_idx_up))
pvalue[de_idx_down] = rng.uniform(1e-12, 0.005, len(de_idx_down))

ranks = np.argsort(np.argsort(pvalue)) + 1
padj  = np.minimum(pvalue * N_GENES / ranks, 1.0)

de_df = pd.DataFrame({
    "contrast":       CONTRAST,
    "gene_id":        gene_ids,
    "gene_name":      gene_names,
    "baseMean":       np.round(baseMean, 3),    # camelCase — matches Streamlit components
    "log2FoldChange": np.round(log2fc, 4),
    "lfcSE":          np.round(lfcse, 4),
    "stat":           np.round(stat, 4),
    "pvalue":         np.round(pvalue, 10),
    "padj":           np.round(padj, 10),
})

n_sig_de = int((de_df["padj"] <= 0.05).sum())
n_up     = int(((de_df["padj"] <= 0.05) & (de_df["log2FoldChange"] > 1)).sum())
n_down   = int(((de_df["padj"] <= 0.05) & (de_df["log2FoldChange"] < -1)).sum())
print(f"DE results : {N_GENES} genes  |  sig (padj≤0.05): {n_sig_de}  |  up: {n_up}  down: {n_down}")

# ── GSEA Results: 25 Reactome pathways ────────────────────────────────────────
PATHWAYS = [
    ("R-HSA-109581",  "Apoptosis",                              180),
    ("R-HSA-1640170", "Cell Cycle",                             488),
    ("R-HSA-5633007", "Regulation of TP53 Degradation",          42),
    ("R-HSA-69278",   "Cell Cycle, Mitotic",                    368),
    ("R-HSA-69205",   "G1/S-Specific Transcription",             41),
    ("R-HSA-5218921", "VEGFA-VEGFR2 Pathway",                   244),
    ("R-HSA-5687128", "MAPK1/MAPK3 Signaling",                  166),
    ("R-HSA-3214815", "HIF-1-alpha Transcription Factor Network", 56),
    ("R-HSA-162582",  "Signal Transduction",                    2677),
    ("R-HSA-1266738", "Developmental Biology",                  1467),
    ("R-HSA-73857",   "RNA Polymerase II Transcription",        659),
    ("R-HSA-212436",  "Generic Transcription Pathway",          534),
    ("R-HSA-168256",  "Immune System",                         2044),
    ("R-HSA-9612973", "Autophagy",                              539),
    ("R-HSA-5663202", "Notch-HLH Transcription Pathway",        23),
    ("R-HSA-400253",  "Circadian Clock",                         30),
    ("R-HSA-389356",  "CD28 Co-stimulation",                     32),
    ("R-HSA-1280218", "Adaptive Immune System",                 737),
    ("R-HSA-8953897", "Cellular Responses to External Stimuli", 595),
    ("R-HSA-9013148", "CDC42 GTPase Cycle",                      86),
    ("R-HSA-9013149", "RAC1 GTPase Cycle",                       90),
    ("R-HSA-2132295", "MHC Class II Antigen Presentation",       74),
    ("R-HSA-6806003", "Regulation of TP53 Expression",           63),
    ("R-HSA-6804756", "Regulation of TP53 Degradation by FBXW7",22),
    ("R-HSA-8878166", "Transcriptional Regulation by Small RNAs",37),
]

nes_vals   = rng.uniform(-2.5, 2.5, len(PATHWAYS))
pvals_gsea = np.clip(rng.exponential(0.08, len(PATHWAYS)), 1e-8, 1.0)
ranks_gsea = np.argsort(np.argsort(pvals_gsea)) + 1
padj_gsea  = np.minimum(pvals_gsea * len(PATHWAYS) / ranks_gsea, 1.0)
leading_edge = [
    ",".join(random.sample(GENE_NAMES, random.randint(4, 10)))
    for _ in PATHWAYS
]

gsea_df = pd.DataFrame({
    "contrast":           CONTRAST,
    "pathway_id":         [p[0] for p in PATHWAYS],
    "pathway":            [p[1] for p in PATHWAYS],   # bubble chart uses 'pathway'
    "pathway_name":       [p[1] for p in PATHWAYS],
    "size":               [p[2] for p in PATHWAYS],
    "NES":                np.round(nes_vals, 4),
    "nes":                np.round(nes_vals, 4),
    "pvalue":             np.round(pvals_gsea, 8),
    "padj":               np.round(padj_gsea, 8),
    "leading_edge_genes": leading_edge,
})

n_sig_gsea = int((gsea_df["padj"] <= 0.05).sum())
print(f"GSEA results: {len(gsea_df)} pathways  |  sig (padj≤0.05): {n_sig_gsea}")

# ── QC Metrics: per-sample FastQC / RSeQC summary ─────────────────────────────
qc_samples = [
    {
        "sample":             name,
        "condition":          cond,
        "total_reads":        random.randint(48_000_000, 55_000_000),
        "pct_gc":             round(random.uniform(47, 55), 1),
        "pct_duplicates":     round(random.uniform(8, 18), 1),
        "mapping_rate":       round(random.uniform(92, 97), 2),
        "pct_exonic":         round(random.uniform(65, 80), 1),
        "pct_intronic":       round(random.uniform(10, 20), 1),
        "pct_intergenic":     round(random.uniform(5, 12), 1),
        "median_insert_size": random.randint(165, 210),
        "n_genes_detected":   random.randint(16_000, 19_500),
    }
    for name, cond, _ in SAMPLES_META
]
qc_metrics = {"samples": qc_samples}

print(f"QC metrics  : {len(qc_samples)} samples")

DE results : 300 genes  |  sig (padj≤0.05): 42  |  up: 22  down: 18
GSEA results: 25 pathways  |  sig (padj≤0.05): 3
QC metrics  : 4 samples


## 6 — Write Streamlit data files

In [7]:
# DE results
de_path = STREAM_DIR / "de_results.csv"
de_df.to_csv(de_path, index=False)

# GSEA results
gsea_path = STREAM_DIR / "gsea_results.csv"
gsea_df.to_csv(gsea_path, index=False)

# QC metrics
qc_path = STREAM_DIR / "qc_metrics.json"
with open(qc_path, "w") as fh:
    json.dump(qc_metrics, fh, indent=2)

# Manifest (tells Streamlit which files are available)
manifest = {
    "run_id":         RUN_ID,
    "run_name":       "treatment_vs_control_demo",
    "pipeline_type":  "bulk_rnaseq",
    "contrast":       CONTRAST,
    "samples":        [s["sample"] for s in qc_samples],
    "available": {
        "de_results":  "de_results.csv",
        "gsea_results": "gsea_results.csv",
        "qc_metrics":  "qc_metrics.json",
    },
}
manifest_path = STREAM_DIR / "manifest.json"
with open(manifest_path, "w") as fh:
    json.dump(manifest, fh, indent=2)

print(f"Streamlit data directory: {STREAM_DIR}")
for f in sorted(STREAM_DIR.iterdir()):
    print(f"  {f.name:<25}  {f.stat().st_size/1024:6.1f} KB")

Streamlit data directory: /Users/junesong/Developer/agent_rnaseq/demo_output/streamlit_data
  de_results.csv               29.6 KB
  gsea_results.csv              4.0 KB
  manifest.json                 0.4 KB
  qc_metrics.json               1.3 KB


## 7 — Inline visualisations

These use the same Plotly components as the Streamlit dashboard.

In [8]:
# ── Volcano plot ──────────────────────────────────────────────────────────────
from src.streamlit.components.volcano_plot import make_volcano_plot

fig_volcano = make_volcano_plot(de_df, padj_cutoff=0.05, lfc_cutoff=1.0)
fig_volcano.update_layout(
    title="Volcano Plot — Treatment vs. Control  (300 genes, DESeq2 Wald test)",
    height=500,
)
fig_volcano.show()

In [9]:
# ── MA plot ───────────────────────────────────────────────────────────────────
from src.streamlit.components.ma_plot import make_ma_plot

fig_ma = make_ma_plot(de_df, padj_cutoff=0.05)
fig_ma.update_layout(
    title="MA Plot — Treatment vs. Control",
    height=450,
)
fig_ma.show()

In [10]:
# ── GSEA pathway bubble chart ─────────────────────────────────────────────────
from src.streamlit.components.pathway_bubble import make_pathway_bubble

fig_gsea = make_pathway_bubble(gsea_df, top_n=20, padj_cutoff=0.25)
fig_gsea.update_layout(
    title="Reactome Pathway Enrichment (bubble size ∝ gene-set size)",
    height=620,
)
fig_gsea.show()

In [11]:
# ── QC metrics table ──────────────────────────────────────────────────────────
from src.streamlit.components.qc_metrics_table import make_qc_table

fig_qc = make_qc_table(qc_metrics)
fig_qc.update_layout(title="QC Metrics — FastQC + RSeQC summary", height=300)
fig_qc.show()

# Pretty-print as a Pandas table too
pd.DataFrame(qc_samples).set_index("sample")

,condition,total_reads,pct_gc,pct_duplicates,mapping_rate,pct_exonic,pct_intronic,pct_intergenic,median_insert_size,n_genes_detected
sample,,,,,,,,,,
ctrl_1,control,48838703,47.6,10.1,93.33,79.0,18.8,11.2,188,17167
ctrl_2,control,49323276,50.5,13.4,93.51,79.8,18.1,8.7,207,19346
treat_1,treatment,52652341,49.4,14.6,96.69,67.0,11.2,5.7,200,16636
treat_2,treatment,50284622,49.3,10.1,93.71,75.3,18.5,8.5,181,19465


## 8 — Top DE genes

In [12]:
sig_de = (
    de_df[de_df["padj"] <= 0.05]
    .sort_values("padj")
    [["gene_name", "baseMean", "log2FoldChange", "padj"]]
    .rename(columns={"log2FoldChange": "log2FC"})
    .head(20)
    .reset_index(drop=True)
)
print(f"Top {len(sig_de)} significant genes (padj ≤ 0.05, sorted by padj):")
sig_de

Top 20 significant genes (padj ≤ 0.05, sorted by padj):


,gene_name,baseMean,log2FC,padj
0,NOTCH1_207,62.635,2.2409,0.001386
1,AKT1_263,1254.892,-2.7577,0.002737
2,CDK4_139,579.103,2.5199,0.002891
3,CTNNB1_148,208.920,1.7120,0.003155
4,CDH1_81,545.598,3.9940,0.003980
5,PTEN_294,923.747,2.3701,0.010727
6,PIK3CA_182,456.782,2.9758,0.011032
7,TYMS_256,108.286,2.3548,0.011346
8,AKT1_216,153.108,-1.7575,0.011726
9,TYMS_259,48.264,2.1871,0.012098


## 9 — Top enriched pathways

In [13]:
sig_gsea = (
    gsea_df[gsea_df["padj"] <= 0.25]
    .sort_values("padj")
    [["pathway_name", "size", "NES", "pvalue", "padj", "leading_edge_genes"]]
    .head(15)
    .reset_index(drop=True)
)
print(f"Top {len(sig_gsea)} enriched pathways (padj ≤ 0.25):")
sig_gsea

Top 15 enriched pathways (padj ≤ 0.25):


,pathway_name,size,NES,pvalue,padj,leading_edge_genes
0,Cellular Responses to External Stimuli,595,0.1023,0.001067,0.026687,"PCNA,MCM2,VEGFA,MYC,NUSAP1,KRAS,CCND1"
1,Apoptosis,180,1.8907,0.003116,0.038944,"PLK1,NF1,STAT3,Ki67,SKP2,MAD2L1,CDC20,CDH1,BUB..."
2,Transcriptional Regulation by Small RNAs,37,0.5936,0.013322,0.047579,"TP53,MAP2K1,TOP2A,E2F1"
3,Cell Cycle,488,-0.6277,0.012585,0.052436,"TS,DHFR,NOTCH1,RAF1,MCM2,MKI67,RB1"
4,Regulation of TP53 Degradation,42,-1.9945,0.017360,0.054251,"HIF1A,EGFR,MDM2,BRCA1,AURKA"
5,HIF-1-alpha Transcription Factor Network,56,0.5361,0.011172,0.055858,"TS,NF1,CTNNB1,JUN"
6,Immune System,2044,1.4738,0.010951,0.068444,"SKP2,NUSAP1,BCL2,VEGFA"
7,VEGFA-VEGFR2 Pathway,244,0.3220,0.045594,0.071241,"TS,CDKN1A,STAT3,Ki67,PTEN,MAD2L1"
8,Adaptive Immune System,737,-2.3801,0.034601,0.072085,"BIRC5,ESR1,MYC,AURKA"
9,Regulation of TP53 Expression,63,0.8417,0.029682,0.074204,"ESR1,ERBB2,STAT3,TS,BUB1,NUSAP1,BCL2,CENPF"


## 10 — Launch Streamlit dashboard

Run the command below in a separate terminal (with the project virtual environment active).
Then open **http://localhost:8501** and set the **Data directory** in the sidebar to the
path shown below.

In [14]:
streamlit_cmd = (
    f"STREAMLIT_DATA_DIR={STREAM_DIR.resolve()} "
    f"streamlit run {PROJECT_ROOT}/src/streamlit/app.py"
)
print("Run this command in a terminal:")
print()
print(f"  {streamlit_cmd}")
print()
print("Then open: http://localhost:8501")
print()
print(f"Data directory to enter in the sidebar:")
print(f"  {STREAM_DIR.resolve()}")
print()
print("Pages available:")
print("  📊 QC Dashboard        — mapping rates, duplication, insert size")
print("  🔬 Differential Expr.  — volcano plot, MA plot, heatmap, DE table")
print("  🧬 Pathway Enrichment  — Reactome GSEA bubble chart")
print("  🔭 Single Cell         — (no scRNA data in this run)")

Run this command in a terminal:

  STREAMLIT_DATA_DIR=/Users/junesong/Developer/agent_rnaseq/demo_output/streamlit_data streamlit run /Users/junesong/Developer/agent_rnaseq/src/streamlit/app.py

Then open: http://localhost:8501

Data directory to enter in the sidebar:
  /Users/junesong/Developer/agent_rnaseq/demo_output/streamlit_data

Pages available:
  📊 QC Dashboard        — mapping rates, duplication, insert size
  🔬 Differential Expr.  — volcano plot, MA plot, heatmap, DE table
  🧬 Pathway Enrichment  — Reactome GSEA bubble chart
  🔭 Single Cell         — (no scRNA data in this run)


In [15]:
# Optional: launch Streamlit as a background subprocess from the notebook
# Uncomment the lines below to start it automatically.

# import subprocess, os
# env = {**os.environ, "STREAMLIT_DATA_DIR": str(STREAM_DIR.resolve())}
# proc = subprocess.Popen(
#     ["streamlit", "run", str(PROJECT_ROOT / "src" / "streamlit" / "app.py"),
#      "--server.headless=true"],
#     env=env,
# )
# print(f"Streamlit PID {proc.pid} — http://localhost:8501")
# # To stop later: proc.terminate()

## 11 — Run summary

In [16]:
from src.db.models.run import AnalysisRun

with Session(engine) as db:
    r = db.get(AnalysisRun, uuid.UUID(RUN_ID))
    print(f"{'─'*60}")
    print(f"Run ID     : {r.id}")
    print(f"Name       : {r.name}")
    print(f"Status     : {r.status.value}")
    print(f"Pipeline   : {r.pipeline_type.value}")
    print(f"Aligner    : {r.aligner.value}")
    print(f"Created    : {r.created_at}")
    print(f"Completed  : {r.completed_at}")
    print()
    print("Results")
    print(f"  DE genes     : {N_GENES} total, {n_sig_de} significant (padj ≤ 0.05)")
    print(f"    Up-regulated  : {n_up}")
    print(f"    Down-regulated: {n_down}")
    print(f"  GSEA pathways: {len(gsea_df)} total, {n_sig_gsea} significant (padj ≤ 0.05)")
    print(f"  QC samples   : {len(qc_samples)}")
    print()
    print(f"Streamlit data: {STREAM_DIR.resolve()}")
    print(f"Database      : {DB_PATH.resolve()}")

────────────────────────────────────────────────────────────
Run ID     : 5a666a5b-8f4b-4878-990a-623eb85fd5bb
Name       : treatment_vs_control_demo
Status     : completed
Pipeline   : bulk_rnaseq
Aligner    : star
Created    : 2026-06-19 04:52:20.728874
Completed  : 2026-06-19 04:52:20.739418

Results
  DE genes     : 300 total, 42 significant (padj ≤ 0.05)
    Up-regulated  : 22
    Down-regulated: 18
  GSEA pathways: 25 total, 3 significant (padj ≤ 0.05)
  QC samples   : 4

Streamlit data: /Users/junesong/Developer/agent_rnaseq/demo_output/streamlit_data
Database      : /Users/junesong/Developer/agent_rnaseq/demo_output/agent_rnaseq_demo.db


---

## What's next?

| Goal | How |
|---|---|
| Run on real FASTQ data | Replace `data/*.fastq.gz` paths with your own files; provide a real STAR/Salmon genome index; set `dry_run=False` |
| Use the REST API | `docker compose -f docker/docker-compose.yml up -d` then `curl -X POST http://localhost:8000/api/v1/runs …` |
| Run on AWS Batch | Set `executor="aws_batch"` and configure `AWS_BATCH_JOB_QUEUE` + `S3_ARTIFACT_BUCKET` |
| Add single-cell data | Use `pipeline_type="scrnaseq"` and register CellRanger BAM/matrix paths |
| View full deployment guide | See `docs/deployment.md` |